# 18. Lecke: MI-ügynökök biztonságossá tétele kriptográfiai nyugtákkal

## Gyakorlati Jegyzetfüzet

Ez a jegyzetfüzet négy gyakorlaton vezet végig:

1. **Írd alá az első nyugtádat** egy ügynök eszközhíváshoz, és ellenőrizd.
2. **Manipuláld a nyugtát**, és nézd meg, hogy az ellenőrzés sikertelen lesz.
3. **Építs három nyugtából álló láncot**, és erősítsd meg a lánc integritását.
4. **Csomagolj be egy Microsoft Agent Framework eszközhívást**, hogy minden művelet nyugtát bocsásson ki.

Minden kriptográfiai primitívát jól karbantartott könyvtárakból importálunk (`pynacl` az Ed25519-hez, `jcs` az RFC 8785 kanonikus JSON-höz, `hashlib` a Python standard könyvtárból SHA-256-hoz). Maga a nyugta logika egyszerű Python, amelyet elolvashatsz és módosíthatsz.

Futtasd a cellákat sorrendben. Minden szakasz rövid és önálló.


## Beállítás

Telepítsd a két függőséget. Mindkettő engedélyező jellegű licenccel rendelkezik (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Segédfüggvények

Ezek a két segédfüggvény a base64url kódolásával (párnázás nélkül) és az SHA-256 kivonatolással foglalkozik tetszőleges objektumok esetén. A jegyzetfüzet többi része így a nyugta logikájára koncentrálhat.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 1. szakasz: Írja alá első visszaigazolását

Képzeljük el, hogy a **Contoso Travel** ügynökünk éppen most keresett járatokat Sydney és Los Angeles között egy ügyfélnek. Rögzíteni szeretnénk a gateway nyilatkozatát erről az eszközhívásról, hogy egy jövőbeni ellenőr külön, kiemelt kibocsátói kulcs alatt ellenőrizhesse azt.

### 1.1 lépés: Aláírási kulcs létrehozása

Éles környezetben az ügynök aláírási kulcsa egy hardveres biztonsági modulban (HSM), Azure Key Vaultban vagy egy hasonló védett tárolóban élne. Ehhez az órához egy friss kulcsot generálunk memóriában.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### 1.2 lépés: Az átvételi bizonylat tartalmának összeállítása

A tartalom minden olyan információt tartalmaz, amit az átvételi bizonylatnak igazolnia kell: ki cselekedett, milyen eszközzel, milyen argumentumokkal, mi érkezett vissza, milyen szabályzat alapján és mikor. Az argumentumokat és az eredményt hash-eljük, ahelyett, hogy beágyaznánk őket, hogy az átvételi bizonylat ne szivárogtasson ki érzékeny tartalmat.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### 1.3. lépés: A nyugta aláírása és összeállítása

Három lépés:

1. Kanonizáld a terhelést JCS segítségével, hogy két megvalósítás ugyanazt a logikai nyugtát bájt azonos bájtokká alakítsa.
2. Írd alá az kanonikus bájtokat közvetlenül az Ed25519 privát kulccsal. A PureEdDSA belsőleg hash-eli az üzenetet, tehát egy extra előhash megváltoztatná a protokollt.

Az aláírást ezután hozzáillesztik az eredeti terheléshez, hogy előállítsák a végleges nyugtát.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### 1.4. lépés: A nyugta ellenőrzése

Az ellenőrzés a folyamat megfordítása. Eltávolítjuk az aláírást, újraszámoljuk a kanonikus bájtokat, és összehasonlítjuk az aláírást a kibocsátó nyilvános kulcsával.

Az ellenőrzés offline zajlik: nincs szükség szolgáltatás hívására vagy hálózati hozzáférésre. Egy dolgot azonban igényel a nyugtán kívül: a kibocsátó nyilvános kulcsát, amelyet az auditor már megbízható csatornán keresztül szerez be, például a kibocsátó által közzétett JWK készletből vagy előre rögzített értékként. A nyugtán belüli `public_key` mező nem játszhatja ezt a szerepet, mert aki a nyugtát írja, az ugyanazt a mezőt is írja. A 2. szekció bemutatja azt a hamisítást, amit ez megakadályoz.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Látnod kell `Receipt is valid: True`, `Pre-hashed receipt valid: False` és `Malformed receipts rejected: True` elemeket. A pozitív eset bizonyítja, hogy a közvetlen-JCS aláírási út működik; a negatív kontroll végrehajthatóvá teszi az aláírás-hatókör szabályt, ahelyett hogy csak szövegként maradna; az utolsó sor hibás bemenetként `False` értéket ad vissza a kivétel dobása helyett.


## 2. szakasz: A nyugta meghamisítása

Az egész célja a nyugtáknak, hogy hamisítás-ellenőrizhetők legyenek. Bizonyítsuk be ezt.

Pontosan egy karaktert fogunk módosítani a nyugtán, és megnézzük, hogy az ellenőrzés sikertelen lesz.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Mi történt most?

Amikor megváltoztattuk a `policy_id` értékét, a kanonikus bájtok is megváltoztak. Az aláírás (amely az eredeti kanonikus bájtokra vonatkozott) már nem egyezik. Az ellenőrzés helyesen `False` értéket ad vissza.

Nincs mód arra, hogy megváltoztassunk egy aláírt mezőt, vagyis bármit a `signature` objektumon kívül, és mégis az ellenőrzés sikeres legyen a kibocsátó nyilvános kulcsával, hacsak a támadónak nincs meg a kibocsátó privát kulcsa. Maga a `signature` objektum nincs aláírva: a `sig` az aláírás, és a `verify_receipt` figyelmen kívül hagyja a `public_key`-t, így csak az adott mező szerkesztése önmagában semmit sem változtat. Amíg a privát kulcs egy kulcstárolóban van, és az ellenőrzők a kibocsátó külön publikált kulcsát használják, egy aláírt mező módosítását lehetetlen eltitkolni. Ez a feltétel számít: a következő cella egy módosított nyugtát mutat, amely tökéletesen ellenőrizhető egy másik kulccsal.

Próbáld ki te magad: módosítsd a `tool_name`-t vagy `agent_id`-at vagy `timestamp`-et a fenti cellában, és futtasd újra. Minden változás egy aláírt mezőben érvénytelen nyugtát eredményez.


### A módosítás, amit az aláírás nem észlel

A fenti támadó nem írta alá újra. Egy körültekintőbb azonban módosítaná a visszaigazolást,
aláírná egy saját maga által generált kulccsal, és azt a kulcsot tenné a
`public_key` mezőbe. Az eredmény egy olyan visszaigazolás, amelynek aláírása teljesen hiteles
a tartalmához.

Futtasd ezt, és nézd meg, hogy a két ellenőrzés közül melyik észleli.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Látnod kell az `True`, majd az `False` kódot. A három `assert` sor azt egy ellenőrzéssé teszi:
a cella hibát dob, ha a hamisítvány valaha is érvényesnek bizonyul az
kibocsátó kulcsa ellen.

A hamisítvány belsőleg konzisztens. A rajta lévő aláírás valóban megegyezik a
tartalmával és a kulccsal, amelyet hordoz, így egy hitelesítő, ami kiolvassa a kulcsot a
bizonylatból, érvényesnek jelenti ki. Semmi sem tűnik hibásnak a bizonylaton, mert
semmi sem ellentmondásos benne.

Az egyetlen dolog, ami elkapja, az az, hogy olyan kulccsal ellenőrzi, amelyre a támadónak nem volt
lehetősége hatni. Ez az oka annak, hogy az `verify_receipt` a kulcsot argumentumként kapja: egy
`verify_receipt(receipt)` aláírású függvény nem írható biztonságosan,
mert az egyetlen kulcs, amelyhez hozzáférhet, az a támadó által megadott.

Ugyanaz a bizonylat `sample_receipts/04_resigned_receipt.json` néven menthető el, ha
meg akarod vizsgálni a jegyzetfüzeten kívül.


## 3. fejezet: Nyugták összefűzése láncba

Egyetlen nyugta egyetlen műveletet véd. A legtöbb ügynök sok műveletet hajt végre. Az egész sorrend manipulációval szembeni védelméhez összekapcsoljuk az egyes nyugtákat az előzővel, úgy, hogy az új nyugta terhelésében szerepel az előző nyugta hash értéke.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ha valaki töröl vagy átrendez egy nyugtát a bemutatott folyamatban, a következő hivatkozás megszakad. A megnyesés vagy aláíró által vezérelt utótag átírásának felismeréséhez is rögzített jelenlegi fejre, külső ellenőrzőpontra vagy hitelesített populációs kötelezettségvállalásra van szükség.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Most szakítsuk meg a láncot azáltal, hogy módosítjuk a középső bizonylatot, és újra ellenőrizzük. A módosított bizonylat nem felel meg az aláírási ellenőrzésnek, ÉS a következő bizonylat sem haladja meg a lánc-lánc ellenőrzést (mert az `previous_receipt_hash` már nem egyezik a módosított középső bizonylat hash-ével).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


A 0. bizonylat még mindig érvényes (nem módosították, és nincs elődje, amire támaszkodhatna). Az 1. bizonylat aláírás-ellenőrzése sikertelen, mert megváltoztattuk `tool_args_hash`-t. A 2. bizonylat lánchivatkozás-ellenőrzése sikertelen, mert `previous_receipt_hash`-ját az eredeti (most módosított) 1. bizonylatra számolták ki.

Még ha egy támadó újraaláírná a módosított 1. bizonylatot, az új aláírás sikertelen az kibocsátó kulcsa ellen, hacsak nem rendelkezik a kibocsátó privát kulcsával (a saját kulccsal történő újraaláírás sikertelen, amint a 2. fejezet bemutatta), és a 2. bizonylat lánchivatkozási eltérése továbbra is felfedné a hamisítást. A változás eltitkolásához a támadónak minden bizonylatot újra kellene aláírnia a módosítási ponttól kezdve, ami a kibocsátó privát kulcsának birtoklását igényli.


## 4. szakasz: Ügynök eszközhívás csomagolása átvétel aláírással

Valódi telepítés esetén nem szeretnéd, hogy minden ügynök szerzője emlékezzen az `make_receipt` hívására. Azt akarod, hogy az átvétel aláírás automatikus legyen minden eszköz meghívásához.

Íme a legegyszerűbb minta: egy csomagoló osztály, amely bármely hívható eszközfüggvényt elfogad, és egy átvételt kibocsátó változatát adja vissza. Ez alkalmazkodik bármely ügynök keretrendszerhez, beleértve a Microsoft Agent Frameworköt (`agent_framework.foundry`) is.

Ha nincs beállítva Microsoft Foundry projekted, az alábbi helyi hamisítás még mindig bemutatja a mintát.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integráció a Microsoft Agent Frameworkkel

A fenti `ReceiptedTool` csomagoló keretfüggetlen. Ahhoz, hogy egy Microsoft Agent Frameworkkel épített ügynökön belül használjuk, regisztráljuk a becsomagolt függvényt eszközként. Egy vázlat (a mockot valódi Microsoft Foundry eszközregisztrációval helyettesítenéd):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Az ügynök keretrendszernek nem kell tudnia semmit a számlákról. A számla aláírása az eszköz köré van becsomagolva, nem a keretrendszerbe integrálva. Így adhatsz hozzá származási információt meglévő ügynökkódhoz anélkül, hogy újraírnád az ügynököt.


## Összefoglalás és bővített kihívás

Önök:

- Ed25519 kulcspárt generáltak.
- Készítettek és aláírtak egy nyugtát egy ügynök eszköz hívásához.
- Offline ellenőrizték a nyugtát kizárólag a kibocsátó nyilvános kulcsával, amelyet a nyugtától függetlenül szereztek be.
- Megváltoztattak egy nyugtát, és megfigyelték az ellenőrzés sikertelenségét.
- Létrehoztak egy három nyugtából álló láncolt hash-sorozatot.
- Megváltoztatták a lánc közepét, és mind az aláírás, mind a lánc-hiba sikertelenségét megfigyelték.
- Egy eszköz funkciót automatikus nyugta aláírással burkoltak be.

**Bővített kihívás.** Bővítse a nyugta sémát egy `request_id` mezővel (egy UUID az elosztott követéshez). Frissítse a `make_receipt`-et úgy, hogy tartalmazza, és igazolja, hogy a nyugták továbbra is végponttól végpontig ellenőrizhetők. Ezután módosítsa a mezőt aláírás után, és igazolja, hogy az ellenőrzés sikertelen lesz. Ez arra kényszeríti, hogy belsőleg értse meg, hogyan járul hozzá a kanonikus kódolás minden byte-ja az aláíráshoz.

**Fontos határ.** Egy rögzített kulcs és lánc kontextus alatt a nyugták bizonyítják az aláírás hozzárendelhetőségét, a tartalom sértetlenségét és a bemutatott előd láncához való kötődést. Nem igazolják a végrehajtást, az eredeti igazságot, a megbízható időt, a teljességet, a szabályzat értékelést vagy a helyességet. A nyugták alapot jelentenek. Az irányítás az a rendszer, amelyet erre épít.

Olvassa újra a lecke README-jét ezt a határt szem előtt tartva. A leggyakoribb hiba, amit a csapatok a nyugtákkal elkövetnek, az az, hogy feltételezik: „van nyugtánk” = „irányítottak vagyunk.” Ez nincs így. A nyugták teszik az ügynök viselkedését auditálhatóvá, de nem teszik helyessé.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
